In [6]:
%%capture
!pip install flask
!pip install pyngrok
!pip install flask_cors


In [7]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [40]:
import os
NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTH_TOKEN")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")


In [41]:
!ngrok config add-authtoken NGROK_AUTH_TOKEN


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [8]:
import getpass
import concurrent.futures
import threading
from multiprocessing import Process
import time
import json
import requests
import wandb

from flask import Flask, request, jsonify
from pyngrok import ngrok, conf
from flask_cors import CORS

from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

In [9]:
from datasets import load_dataset


In [11]:
def initialize_model(model_name):
  model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
  )
  model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
  )
  return model, tokenizer


In [12]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples, EOS_TOKEN):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [13]:
def finetune_model(model_, batch_size, grad_steps, epochs, learning_rate, metric):
    print("in finetune model")
    model, tokenizer = initialize_model(model_)

    eos = tokenizer.eos_token
    ee_dataset = load_dataset("tayyibsupercool/resource_allocation_telecom_energy_efficiency_instruct", split = "train[:100]")
    ee_dataset = ee_dataset.map(lambda examples: formatting_prompts_func(examples, eos), batched=True)
    se_dataset = load_dataset("tayyibsupercool/resource_allocation_telecom_spectral_efficiency_instruct", split = "train[:100]")
    se_dataset = se_dataset.map(lambda examples: formatting_prompts_func(examples, eos), batched=True)


    dataset_ = ee_dataset if metric == "Energy Efficiency" else se_dataset

    # dataset = dataset.map(formatting_prompts_func, batched = True,)

    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = dataset_,
        dataset_text_field = "text",
        max_seq_length = 2048,
        dataset_num_proc = 2,
        packing = False, # Can make training 5x faster for short sequences.
        args = TrainingArguments(
            per_device_train_batch_size = int(batch_size), # 2 to 8
            gradient_accumulation_steps = int(grad_steps), # 2 to 8
            warmup_steps = 5,
            # num_train_epochs = epochs, # Set this for 1 full training run.
            max_steps = 60,
            learning_rate = float(learning_rate),
            fp16 = not is_bfloat16_supported(),
            bf16 = is_bfloat16_supported(),
            logging_steps = 1,
            optim = "adamw_8bit",
            weight_decay = 0.01,
            lr_scheduler_type = "linear",
            seed = 3407,
            output_dir = "outputs",
        ),
    )


    print("Starting fine-tuning...", flush=True)
    # response = requests.post("http://127.0.0.1:5000/finetuning_started", json={"message": f"Finetuning of {model} has started and will take approximately 3 minutes to complete."})

    trainer.train()


In [34]:
!pkill -f ngrok

In [36]:
# print("Enter your authtoken, which can be copied from https://dashboard.ngrok.com/get-started/your-authtoken")
# conf.get_default().auth_token = getpass.getpass()

app = Flask(__name__)
CORS(app)
port = "5001"

# Open a ngrok tunnel to the HTTP server
public_url = ngrok.connect(port).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")

# Update any base URLs to use the public ngrok URL
app.config["BASE_URL"] = public_url

# ... Update inbound traffic via APIs to use the public-facing ngrok URL

# threading.Thread(target=app.run, kwargs={"host": "0.0.0.0", "port": int(port), "use_reloader": False}).start()

executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)  # Global executor

def run_flask():
    app.run(host="0.0.0.0", port=int(port), use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()  # Start Flask in a separate thread


# 2u1ouXLUelqJM8CKhTJKbtNjlYu_62beWTsPGeUQyY1V3hY22 (ngrok)

 * ngrok tunnel "https://1c1a-34-145-93-8.ngrok-free.app" -> "http://127.0.0.1:5001"


In [16]:
@app.route("/start_finetuning", methods=["POST"])
def begin_finetuning():

    try:
        data = request.get_json()  # Ensure JSON parsing
        if not data:
            return jsonify({"error": "Invalid JSON payload"}), 400
        print(data)
        print("Raw model_name:", data.get("model"))
        model_name = str(data.get("model"))  # Using `.get()` to avoid KeyError
        batch_size = int(data.get("batch_size", 32))
        grad_steps = int(data.get("grad_steps", 1))
        learning_rate = float(data.get("learning_rate", 2e-5))
        epochs = int(data.get("epochs", 1))
        dataset = data.get("dataset", "Energy Efficiency")
        user = data.get("user", "unknown")
        print("model_name:", model_name, flush=True)
        print(f"Received hyperparameters: Epochs={grad_steps, epochs, dataset}, Batch Size={batch_size}, Learning Rate={learning_rate}")

        finetune_model(
            model_=model_name,
            batch_size=batch_size,
            grad_steps=grad_steps,
            learning_rate=learning_rate,
            epochs=epochs,
            metric=dataset
        )

        return jsonify({"message": "Finetuning complete! You may now chat with the LLM trained to predict optimal transmit powers!"}), 200

    except Exception as e:
        print("Error in fine-tuning:", str(e))
        return jsonify({"error": str(e)}), 500


# Define Flask routes
@app.route("/")
def index():
    return "Hello from Colab!"


In [42]:
wandb.login(key=WANDB_API_KEY)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [18]:
!curl -X POST https://d51c-34-145-93-8.ngrok-free.app/start_finetuning -H "Content-Type: application/json" -d '{"model": "unsloth/Phi-3.5-mini-instruct", "batch_size": "32", "learning_rate": "2e-5", "epochs": "1", "user": "test"}'


{'model': 'unsloth/Phi-3.5-mini-instruct', 'batch_size': '32', 'learning_rate': '2e-5', 'epochs': '1', 'user': 'test'}
Raw model_name: unsloth/Phi-3.5-mini-instruct
model_name: unsloth/Phi-3.5-mini-instruct
Received hyperparameters: Epochs=(1, 1, 'Energy Efficiency'), Batch Size=32, Learning Rate=2e-05
in finetune model
==((====))==  Unsloth 2025.3.9: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

^C


Confirming connections

In [21]:
!curl -I https://d51c-34-145-93-8.ngrok-free.app/start_finetuning


HTTP/1.1 404 Not Found
Connection: close
Content-Type: text/html
Ngrok-Error-Code: ERR_NGROK_3200
Referrer-Policy: no-referrer
Date: Tue, 11 Mar 2025 14:01:38 GMT



In [22]:
!curl http://127.0.0.1:5001


INFO:werkzeug:127.0.0.1 - - [11/Mar/2025 14:02:39] "GET / HTTP/1.1" 200 -


Hello from Colab!

In [29]:
!ps aux | grep flask


root        8325  0.0  0.0   7376  3404 ?        S    14:05   0:00 /bin/bash -c ps aux | grep flask
root        8327  0.0  0.0   6484  2260 ?        S    14:05   0:00 grep flask


In [37]:
!curl http://localhost:4040/api/tunnels


{"tunnels":[{"name":"http-5001-911dd1d4-6558-48bf-8360-40159dee38c4","ID":"f13d348074fd372c84dd1a39155f7a37","uri":"/api/tunnels/http-5001-911dd1d4-6558-48bf-8360-40159dee38c4","public_url":"https://1c1a-34-145-93-8.ngrok-free.app","proto":"https","config":{"addr":"http://localhost:5001","inspect":true},"metrics":{"conns":{"count":0,"gauge":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0},"http":{"count":0,"rate1":0,"rate5":0,"rate15":0,"p50":0,"p90":0,"p95":0,"p99":0}}}],"uri":"/api/tunnels"}
